In [4]:
import json
from pathlib import Path
import pandas as pd

In [2]:
with open("/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/molecule_safe_ver/QA/LLMs/safe_qa_outputs/task1_safe_to_nontoxic/multi_step/summary_gpt-4o.json", "r", encoding="utf-8") as f:
    task1_gpt4o_base = json.load(f)

In [3]:
task1_gpt4o_base

{'task': 1,
 'variant': 'base',
 'model': 'gpt-4o',
 'total': 20,
 'correct': 0,
 'accuracy': 0.0,
 'metrics_mean': {'fragment_EM': 0.0,
  'fragment_BLEU1': 0.08815371762740185,
  'fragment_Precision': 0.0,
  'fragment_Recall': 0.0,
  'fragment_F1': 0.0,
  'molecule_EM': 0.0,
  'molecule_morganFT': 0.13313848951176527,
  'molecule_validity': 0.7}}

In [9]:
import json
import re
from pathlib import Path
import pandas as pd

# summary json들이 있는 상위 폴더
base_dir = Path("/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/molecule_safe_ver/QA/LLMs/safe_qa_outputs")

rows = []

# 하위 폴더까지 전부 뒤져서 summary_*.json 찾기
for file_path in base_dir.rglob("summary_*.json"):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        metrics = data.get("metrics_mean", {})

        file_name = file_path.name

        # run1, run2 같은 값 추출
        run_match = re.search(r"run(\d+)", file_name)
        run_index = int(run_match.group(1)) if run_match else None

        # icl1, icl2 같은 값도 필요하면 같이 추출
        icl_match = re.search(r"icl(\d+)", file_name)
        icl_index = int(icl_match.group(1)) if icl_match else None

        rows.append({
            "file_name": file_name,
            "task_dir": file_path.parent.parent.name,   # ex) task1_safe_to_nontoxic
            "step_type": file_path.parent.name,         # ex) multi_step / single_step
            "run_index": run_index,
            "icl_index": icl_index,
            "task": data.get("task"),
            "variant": data.get("variant"),
            "model": data.get("model"),
            "total": data.get("total"),
            "correct": data.get("correct"),
            "accuracy": data.get("accuracy"),
            "fragment_EM": metrics.get("fragment_EM"),
            "fragment_BLEU1": metrics.get("fragment_BLEU1"),
            "fragment_Precision": metrics.get("fragment_Precision"),
            "fragment_Recall": metrics.get("fragment_Recall"),
            "fragment_F1": metrics.get("fragment_F1"),
            "molecule_EM": metrics.get("molecule_EM"),
            "molecule_morganFT": metrics.get("molecule_morganFT"),
            "molecule_validity": metrics.get("molecule_validity"),
            "full_path": str(file_path)
        })

    except Exception as e:
        print(f"읽기 실패: {file_path} | {e}")

# DataFrame 생성
df_compare = pd.DataFrame(rows)

# 보기 좋게 정렬
df_compare = df_compare.sort_values(
    by=["task", "step_type", "model", "variant", "icl_index", "run_index", "file_name"],
    ascending=[True, True, True, True, True, True, True]
).reset_index(drop=True)

In [10]:
print(df_compare.columns)
df_compare.head()

Index(['file_name', 'task_dir', 'step_type', 'run_index', 'icl_index', 'task',
       'variant', 'model', 'total', 'correct', 'accuracy', 'fragment_EM',
       'fragment_BLEU1', 'fragment_Precision', 'fragment_Recall',
       'fragment_F1', 'molecule_EM', 'molecule_morganFT', 'molecule_validity',
       'full_path'],
      dtype='object')


,file_name,task_dir,step_type,run_index,icl_index,task,variant,model,total,correct,accuracy,fragment_EM,fragment_BLEU1,fragment_Precision,fragment_Recall,fragment_F1,molecule_EM,molecule_morganFT,molecule_validity,full_path
0,summary_gpt-4o_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,NaN,1,base,gpt-4o,20,3,0.15,0.15,0.903000,0.912500,0.650000,0.726190,0.0,0.250346,0.90,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
1,summary_gpt-4o.json,task1_safe_to_nontoxic,multi_step,NaN,NaN,1,base,gpt-4o,20,0,0.00,0.00,0.088154,0.000000,0.000000,0.000000,0.0,0.133138,0.70,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
2,summary_gpt-4o_icl1_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,1.0,1,icl1,gpt-4o,20,5,0.25,0.25,0.947787,0.951681,0.751283,0.785821,0.5,0.760743,1.00,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
3,summary_gpt-4o_icl1_task1.json,task1_safe_to_nontoxic,multi_step,NaN,1.0,1,icl1,gpt-4o,20,0,0.00,0.00,0.428513,0.040476,0.039216,0.039785,0.0,0.338023,0.85,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
4,summary_gpt-4o_icl2_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,2.0,1,icl2,gpt-4o,20,5,0.25,0.25,0.929491,0.967173,0.834381,0.886523,0.5,0.773977,1.00,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...


In [ ]:
multi_step_df = df_compare[df_compare['step_type'] == "multi_step"]
mutli_step_df_1 = multi_step_df[multi_step_df['task'] == 1]
mutli_step_df_3 = multi_step_df[multi_step_df['task'] == 3]
mutli_step_df_4 = multi_step_df[multi_step_df['task'] == 4]
single_step_df = df_compare[df_compare['step_type'] == "single_step"]
sinlge_step_df_1 = single_step_df[single_step_df['task'] == 1]
sinlge_step_df_3 = single_step_df[single_step_df['task'] == 3]
sinlge_step_df_4 = single_step_df[single_step_df['task'] == 4]

run1_df = df_compare[df_compare['run_index'] == 1]

In [ ]:
run1_dfs = {}

for tasks in run1_df['task'].unique():
    run1_dfs[f'{tasks}'] = run1_df[run1_df['task'] == tasks]

In [31]:
run1_dfs['4']

,file_name,task_dir,step_type,run_index,icl_index,task,variant,model,total,correct,accuracy,fragment_EM,fragment_BLEU1,fragment_Precision,fragment_Recall,fragment_F1,molecule_EM,molecule_morganFT,molecule_validity,full_path
36,summary_gpt-4o_run1_task4.json,task4_safe_to_nontoxic_smiles,multi_step,1.0,NaN,4,base,gpt-4o,20,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
38,summary_gpt-4o_icl1_run1_task4.json,task4_safe_to_nontoxic_smiles,multi_step,1.0,1.0,4,icl1,gpt-4o,20,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
40,summary_gpt-4o_icl2_run1_task4.json,task4_safe_to_nontoxic_smiles,multi_step,1.0,2.0,4,icl2,gpt-4o,20,1,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
42,summary_gpt-4o_icl4_run1_task4.json,task4_safe_to_nontoxic_smiles,multi_step,1.0,4.0,4,icl4,gpt-4o,20,1,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
44,summary_gpt-4o_run1_task4.json,task4_safe_to_nontoxic_smiles,single_step,1.0,NaN,4,base,gpt-4o,20,2,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
46,summary_gpt-4o_icl1_run1_task4.json,task4_safe_to_nontoxic_smiles,single_step,1.0,1.0,4,icl1,gpt-4o,20,2,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
48,summary_gpt-4o_icl2_run1_task4.json,task4_safe_to_nontoxic_smiles,single_step,1.0,2.0,4,icl2,gpt-4o,20,3,0.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
50,summary_gpt-4o_icl4_run1_task4.json,task4_safe_to_nontoxic_smiles,single_step,1.0,4.0,4,icl4,gpt-4o,20,2,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...


In [22]:
sinlge_step_df_1.head(2)

,file_name,task_dir,step_type,run_index,icl_index,task,variant,model,total,correct,accuracy,fragment_EM,fragment_BLEU1,fragment_Precision,fragment_Recall,fragment_F1,molecule_EM,molecule_morganFT,molecule_validity,full_path
8,summary_gpt-4o_run1_task1.json,task1_safe_to_nontoxic,single_step,1.0,NaN,1,base,gpt-4o,20,19,0.95,0.95,0.900000,0.95,0.95,0.95,0.05,0.231643,0.9,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
9,summary_gpt-4o.json,task1_safe_to_nontoxic,single_step,NaN,NaN,1,base,gpt-4o,20,0,0.00,0.00,0.170417,0.00,0.00,0.00,0.00,0.345911,1.0,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...


In [19]:
mutli_step_df_1.head(2)

,file_name,task_dir,step_type,run_index,icl_index,task,variant,model,total,correct,accuracy,fragment_EM,fragment_BLEU1,fragment_Precision,fragment_Recall,fragment_F1,molecule_EM,molecule_morganFT,molecule_validity,full_path
0,summary_gpt-4o_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,NaN,1,base,gpt-4o,20,3,0.15,0.15,0.903000,0.9125,0.65,0.72619,0.0,0.250346,0.9,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
1,summary_gpt-4o.json,task1_safe_to_nontoxic,multi_step,NaN,NaN,1,base,gpt-4o,20,0,0.00,0.00,0.088154,0.0000,0.00,0.00000,0.0,0.133138,0.7,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...


In [16]:
print(multi_step_df['task'].unique())
multi_step_df.head()

[1 3 4]


,file_name,task_dir,step_type,run_index,icl_index,task,variant,model,total,correct,accuracy,fragment_EM,fragment_BLEU1,fragment_Precision,fragment_Recall,fragment_F1,molecule_EM,molecule_morganFT,molecule_validity,full_path
0,summary_gpt-4o_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,NaN,1,base,gpt-4o,20,3,0.15,0.15,0.903000,0.912500,0.650000,0.726190,0.0,0.250346,0.90,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
1,summary_gpt-4o.json,task1_safe_to_nontoxic,multi_step,NaN,NaN,1,base,gpt-4o,20,0,0.00,0.00,0.088154,0.000000,0.000000,0.000000,0.0,0.133138,0.70,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
2,summary_gpt-4o_icl1_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,1.0,1,icl1,gpt-4o,20,5,0.25,0.25,0.947787,0.951681,0.751283,0.785821,0.5,0.760743,1.00,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
3,summary_gpt-4o_icl1_task1.json,task1_safe_to_nontoxic,multi_step,NaN,1.0,1,icl1,gpt-4o,20,0,0.00,0.00,0.428513,0.040476,0.039216,0.039785,0.0,0.338023,0.85,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
4,summary_gpt-4o_icl2_run1_task1.json,task1_safe_to_nontoxic,multi_step,1.0,2.0,1,icl2,gpt-4o,20,5,0.25,0.25,0.929491,0.967173,0.834381,0.886523,0.5,0.773977,1.00,/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/mo...
